<a href="https://colab.research.google.com/github/oojobe/FB-/blob/main/Group4Project_Traffic_DataWrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Emergency Dispatch and Traffic Physics Dataset — Data Wrangling

## NPower Group 4 Data Analytics Project

This notebook documents the data-wrangling process for the Emergency Dispatch and Traffic Physics dataset used by Group 4.

The purpose of this stage is to prepare a clean, consistent, and analysis-ready dataset for later exploratory data analysis, visualization, and dashboard development.

### Data Wrangling Objectives
- Load and inspect the source dataset
- Identify missing values and determine whether they are truly missing or structurally not applicable
- Standardize inconsistent categorical values
- Remove duplicate records
- Correct data types
- Impute missing numeric values using context-appropriate methods
- Detect and correct invalid numeric values
- Validate relationships between related fields
- Perform feature engineering
- Export and verify the cleaned dataset

Data Source: https://www.kaggle.com/datasets/razanihababdellatif/emergency-dispatch-and-traffic-physics-dataset

In [ ]:
import pandas as pd

## 1. Load the Dataset

The source CSV file is loaded into a Pandas DataFrame. The original dataset is initially left unchanged so that its structure and data quality can be inspected before any cleaning operations are applied.

In [ ]:
df = pd.read_csv("synthetic_traffic_benchmark_200k.csv")

## 2. Initial Dataset Inspection

The dataset is inspected to understand its structure before making any changes.

The following checks examine:
- sample records
- dataset dimensions
- column data types
- non-null counts
- missing values

This establishes the baseline condition of the raw dataset.

In [ ]:
df.head()

,Record_ID,Timestamp,Date,Hour,Month,Day_of_Week,Time_of_Day,Is_Holiday,Area_Type,Road_Type,...,Vehicle_Age_yrs,Recorded_Speed_kmh,Baseline_Expected_Response_Time_min,Accident_Occurred,Collision_Type,Vehicles_Involved,Accident_Severity,Dispatch_Priority,Actual_Response_Time_min,Road_Closure_Duration_min
0,REC_0000000,2024-03-14 17:35:06.841829282,2024-03-14,17,3,Thursday,Evening Peak,0,Urban,Interstate,...,7.2,49.0,6.2,0,NaN,0,No Accident,NaN,0.0,0.0
1,REC_0000001,2023-05-25 08:10:56.398060108,2023-05-25,8,5,Thursday,Morning Peak,0,Suburban,Interstate,...,9.4,70.1,15.0,0,NaN,0,No Accident,NaN,0.0,0.0
2,REC_0000002,2023-11-07 08:25:17.721608262,2023-11-07,8,11,Tuesday,Morning Peak,0,Suburban,Interstate,...,2.9,71.9,14.6,0,NaN,0,No Accident,NaN,0.0,0.0
3,REC_0000003,2024-07-24 09:30:00.947374674,2024-07-24,9,7,Wednesday,Morning Peak,0,Suburban,Collector,...,4.4,58.7,13.2,0,NaN,0,No Accident,NaN,0.0,0.0
4,REC_0000004,2024-04-03 09:34:07.316651267,2024-04-03,9,4,Wednesday,Morning Peak,0,Urban,Local,...,1.8,25.4,7.8,0,NaN,0,No Accident,NaN,0.0,0.0


In [ ]:
df.shape

(200000, 29)

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 29 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   Record_ID                            200000 non-null  str    
 1   Timestamp                            200000 non-null  str    
 2   Date                                 200000 non-null  str    
 3   Hour                                 200000 non-null  int64  
 4   Month                                200000 non-null  int64  
 5   Day_of_Week                          200000 non-null  str    
 6   Time_of_Day                          200000 non-null  str    
 7   Is_Holiday                           200000 non-null  int64  
 8   Area_Type                            200000 non-null  str    
 9   Road_Type                            200000 non-null  str    
 10  Intersection_Type                    131481 non-null  str    
 11  Speed_Limit_kmh         

In [ ]:
df.isna().sum().sort_values(ascending=False)

Dispatch_Priority                      191769
Collision_Type                         191769
Intersection_Type                       68519
Visibility_m                             5093
Driver_Experience_yrs                    3996
Month                                       0
Day_of_Week                                 0
Is_Holiday                                  0
Time_of_Day                                 0
Timestamp                                   0
Date                                        0
Hour                                        0
Record_ID                                   0
Weather                                     0
Speed_Limit_kmh                             0
Road_Type                                   0
Area_Type                                   0
Vehicle_Type                                0
Driver_Age                                  0
Congestion_Index                            0
Road_Condition                              0
Recorded_Speed_kmh                

## 3. Investigate Missing Values

Missing values are examined according to their context rather than being automatically removed or filled.

Several fields contain missing values, including accident-related categories, intersection type, visibility, and driver experience. The next checks determine whether these missing values represent missing information or fields that are logically not applicable.

In [ ]:
df.groupby("Accident_Occurred").agg(
    Rows=("Accident_Occurred", "size"),
    Missing_Collision_Type=("Collision_Type", lambda x: x.isna().sum()),
    Missing_Dispatch_Priority=("Dispatch_Priority", lambda x: x.isna().sum())
)

,Rows,Missing_Collision_Type,Missing_Dispatch_Priority
Accident_Occurred,,,
0,191769,191769,191769
1,8231,0,0


In [ ]:
pd.crosstab(
    df["Road_Type"],
    df["Intersection_Type"].fillna("Missing")
)

Intersection_Type,4-Way,Missing,Roundabout,T-Junction
Road_Type,,,,
Arterial,34046,0,13386,20177
Collector,14598,7317,0,14442
Interstate,0,38058,0,0
Local,17520,23144,0,17312


In [ ]:
intersection_missing = (
    df.groupby("Road_Type")["Intersection_Type"]
      .agg(
          Total_Rows="size",
          Missing=lambda x: x.isna().sum()
      )
)

intersection_missing["Missing_Percent"] = (
    intersection_missing["Missing"] /
    intersection_missing["Total_Rows"] * 100
)

intersection_missing

,Total_Rows,Missing,Missing_Percent
Road_Type,,,
Arterial,67609,0,0.000000
Collector,36357,7317,20.125423
Interstate,38058,38058,100.000000
Local,57976,23144,39.919967


In [ ]:
visibility_missing = (
    df.groupby("Weather")["Visibility_m"]
      .agg(
          Total_Rows="size",
          Missing=lambda x: x.isna().sum()
      )
)

visibility_missing["Missing_Percent"] = (
    visibility_missing["Missing"] /
    visibility_missing["Total_Rows"] * 100
)

visibility_missing.sort_values(
    "Missing_Percent",
    ascending=False
)

,Total_Rows,Missing,Missing_Percent
Weather,,,
RAIN,1467,100,6.816633
Rain,39615,2414,6.093651
Snow,23073,1386,6.007021
rain,1464,85,5.806011
Rain,1470,85,5.782313
Fog,17771,1023,5.756570
Clear,115140,0,0.000000


### 3.1 Structural Missing Values

The missing-value patterns show that some null values are structurally meaningful.

- `Collision_Type` and `Dispatch_Priority` are missing when no accident occurred.
- `Intersection_Type` is not applicable to all roadway records, particularly Interstate records and some road segments.
- `Visibility_m` contains genuinely missing numeric observations that require later imputation.

These fields should therefore be handled differently rather than using a single missing-value strategy for the entire dataset.

## 4. Inspect Categorical Data Quality

Categorical fields are reviewed for inconsistent capitalization, whitespace, spelling, and unexpected values.

The `Weather` column contains multiple representations of the same condition, including variations of `Rain` with different capitalization and whitespace. These values must be standardized before grouping or analysis.

In [ ]:
weather_counts = df["Weather"].value_counts(dropna=False)

for weather, count in weather_counts.items():
    print(repr(weather), count)

'Clear' 115140
'Rain' 39615
'Snow' 23073
'Fog' 17771
'Rain ' 1470
' RAIN' 1467
'rain' 1464


In [ ]:
categorical_columns = [
    col for col in df.select_dtypes(include="str").columns
    if df[col].nunique(dropna=False) <= 50
]

for col in categorical_columns:
    print(f"\n{col}:")
    print([repr(value) for value in df[col].drop_duplicates()])


Day_of_Week:
["'Thursday'", "'Tuesday'", "'Wednesday'", "'Friday'", "'Saturday'", "'Sunday'", "'Monday'"]

Time_of_Day:
["'Evening Peak'", "'Morning Peak'", "'Night'", "'Midday'"]

Area_Type:
["'Urban'", "'Suburban'", "'Rural'"]

Road_Type:
["'Interstate'", "'Collector'", "'Local'", "'Arterial'"]

Intersection_Type:
['nan', "'T-Junction'", "'4-Way'", "'Roundabout'"]

Weather:
["'Clear'", "'Rain'", "'Rain '", "'Fog'", "'Snow'", "' RAIN'", "'rain'"]

Road_Condition:
["'Dry'", "'Wet'", "'Snowy'", "'Icy'", "'Standing Water'"]

Vehicle_Type:
["'SUV'", "'Sedan'", "'Truck'", "'Motorcycle'", "'Bus'"]

Collision_Type:
['nan', "'Rear-End'", "'Side-Impact'", "'Single-Vehicle'", "'Head-On'"]

Accident_Severity:
["'No Accident'", "'Minor'", "'Severe'", "'Moderate'", "'Fatal'"]

Dispatch_Priority:
['nan', "'Standard'", "'High'", "'Medium'"]


In [ ]:
for col in ["Collision_Type", "Accident_Severity", "Dispatch_Priority"]:
    print(f"\n{col}:")
    print([repr(value) for value in df[col].drop_duplicates()])


Collision_Type:
['nan', "'Rear-End'", "'Side-Impact'", "'Single-Vehicle'", "'Head-On'"]

Accident_Severity:
["'No Accident'", "'Minor'", "'Severe'", "'Moderate'", "'Fatal'"]

Dispatch_Priority:
['nan', "'Standard'", "'High'", "'Medium'"]


## 5. Create Raw and Working Copies

The original DataFrame is preserved as `df_raw`, while all cleaning operations are performed on a separate `df_clean` copy.

This preserves the source data and provides a clear audit trail between the raw and cleaned datasets.

In [ ]:
df_raw = df.copy()
df_clean = df.copy()

## 6. Identify and Remove Duplicate Records

Exact duplicate records are identified before any imputation is performed so that duplicated observations do not influence calculated summary statistics.

Duplicate records are inspected using both full-row duplication checks and repeated `Record_ID` values. One copy of each duplicated record is retained.

In [ ]:
df_clean.duplicated().sum()

np.int64(597)

In [ ]:
duplicate_rows = df_clean[df_clean.duplicated(keep=False)]

duplicate_rows.head(10)

,Record_ID,Timestamp,Date,Hour,Month,Day_of_Week,Time_of_Day,Is_Holiday,Area_Type,Road_Type,...,Vehicle_Age_yrs,Recorded_Speed_kmh,Baseline_Expected_Response_Time_min,Accident_Occurred,Collision_Type,Vehicles_Involved,Accident_Severity,Dispatch_Priority,Actual_Response_Time_min,Road_Closure_Duration_min
1151,REC_0001151,2024-05-17 09:41:21.646259533,2024-05-17,9,5,Friday,Morning Peak,0,Urban,Local,...,6.5,22.6,6.6,0,NaN,0,No Accident,NaN,0.0,0.0
1154,REC_0001154,2024-01-31 06:02:31.792828001,2024-01-31,6,1,Wednesday,Morning Peak,0,Suburban,Local,...,3.7,34.8,9.5,0,NaN,0,No Accident,NaN,0.0,0.0
2037,REC_0002037,2022-03-01 08:00:51.357037201,2022-03-01,8,3,Tuesday,Morning Peak,0,Suburban,Arterial,...,5.7,43.7,9.7,0,NaN,0,No Accident,NaN,0.0,0.0
2207,REC_0002207,2023-03-22 16:59:17.218541076,2023-03-22,16,3,Wednesday,Evening Peak,0,Suburban,Arterial,...,14.5,40.2,11.2,0,NaN,0,No Accident,NaN,0.0,0.0
2289,REC_0002289,2022-10-20 15:03:58.098088511,2022-10-20,15,10,Thursday,Midday,0,Urban,Arterial,...,6.9,44.6,7.3,0,NaN,0,No Accident,NaN,0.0,0.0
2306,REC_0002306,2022-10-23 12:58:24.212548488,2022-10-23,12,10,Sunday,Midday,0,Suburban,Collector,...,5.8,36.1,13.3,0,NaN,0,No Accident,NaN,0.0,0.0
2469,REC_0002469,2023-05-09 15:12:38.922086870,2023-05-09,15,5,Tuesday,Midday,0,Suburban,Local,...,1.7,32.8,12.0,0,NaN,0,No Accident,NaN,0.0,0.0
2763,REC_0002763,2025-07-28 19:27:21.105322408,2025-07-28,19,7,Monday,Evening Peak,0,Urban,Local,...,3.7,36.4,7.5,0,NaN,0,No Accident,NaN,0.0,0.0
2935,REC_0002935,2025-10-25 17:49:27.993756727,2025-10-25,17,10,Saturday,Evening Peak,0,Suburban,Collector,...,4.0,45.4,10.1,0,NaN,0,No Accident,NaN,0.0,0.0
2958,REC_0002958,2023-08-13 13:42:04.824284904,2023-08-13,13,8,Sunday,Midday,0,Urban,Local,...,22.3,33.4,10.3,0,NaN,0,No Accident,NaN,0.0,0.0


In [ ]:
duplicate_rows["Record_ID"].value_counts().head(20)

Record_ID
REC_0001151    2
REC_0001154    2
REC_0002037    2
REC_0002207    2
REC_0002289    2
REC_0002306    2
REC_0002469    2
REC_0002763    2
REC_0002935    2
REC_0002958    2
REC_0003169    2
REC_0003335    2
REC_0003411    2
REC_0003768    2
REC_0004478    2
REC_0005705    2
REC_0008055    2
REC_0009254    2
REC_0009270    2
REC_0009304    2
Name: count, dtype: int64

In [ ]:
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

In [ ]:
df_clean.duplicated().sum()

np.int64(0)

## 7. Clean Categorical Values

Inconsistent `Weather` values are standardized by removing leading and trailing whitespace and applying consistent capitalization.

Structurally missing categorical fields are also converted into explicit categories:

- `Collision_Type` → `No Accident`
- `Dispatch_Priority` → `Not Applicable`
- `Intersection_Type` → `No Intersection`

This preserves valid records while making the meaning of previously missing values explicit.

In [ ]:
df_clean["Weather"] = (
    df_clean["Weather"]
    .str.strip()
    .str.title()
)

In [ ]:
df_clean["Weather"].value_counts()

Weather
Clear    114802
Rain      43883
Snow      23003
Fog       17715
Name: count, dtype: int64

In [ ]:
df_clean["Collision_Type"] = (
    df_clean["Collision_Type"]
    .fillna("No Accident")
)

df_clean["Dispatch_Priority"] = (
    df_clean["Dispatch_Priority"]
    .fillna("Not Applicable")
)

df_clean["Intersection_Type"] = (
    df_clean["Intersection_Type"]
    .fillna("No Intersection")
)

In [ ]:
df_clean[
    ["Collision_Type", "Dispatch_Priority", "Intersection_Type"]
].isna().sum()

Collision_Type       0
Dispatch_Priority    0
Intersection_Type    0
dtype: int64

In [ ]:
for col in [
    "Collision_Type",
    "Dispatch_Priority",
    "Intersection_Type"
]:
    print(f"\n{col}:")
    print(df_clean[col].value_counts())


Collision_Type:
Collision_Type
No Accident       191190
Rear-End            4893
Side-Impact         1591
Single-Vehicle      1488
Head-On              241
Name: count, dtype: int64

Dispatch_Priority:
Dispatch_Priority
Not Applicable    191190
Standard            4115
High                2550
Medium              1548
Name: count, dtype: int64

Intersection_Type:
Intersection_Type
No Intersection    68299
4-Way              65971
T-Junction         51773
Roundabout         13360
Name: count, dtype: int64


In [ ]:
df_clean.isna().sum().sort_values(ascending=False)

Visibility_m                           5079
Driver_Experience_yrs                  3986
Date                                      0
Timestamp                                 0
Record_ID                                 0
Day_of_Week                               0
Time_of_Day                               0
Is_Holiday                                0
Area_Type                                 0
Road_Type                                 0
Intersection_Type                         0
Hour                                      0
Month                                     0
Weather                                   0
Speed_Limit_kmh                           0
Congestion_Index                          0
Road_Condition                            0
Vehicle_Type                              0
Driver_Age                                0
Vehicle_Age_yrs                           0
Recorded_Speed_kmh                        0
Baseline_Expected_Response_Time_min       0
Accident_Occurred               

## 8. Impute Missing Visibility Values

`Visibility_m` contains missing numeric values.

Because visibility varies substantially by weather condition, a single global median would not represent the data appropriately. Missing visibility values are therefore imputed using the median visibility within each weather category.

This preserves the relationship between weather conditions and expected visibility while reducing the influence of extreme values.

In [ ]:
visibility_summary = (
    df_clean.groupby("Weather")["Visibility_m"]
    .agg(
        Total_Rows="size",
        Missing=lambda x: x.isna().sum(),
        Mean="mean",
        Median="median",
        Min="min",
        Max="max"
    )
)

visibility_summary["Missing_Percent"] = (
    visibility_summary["Missing"]
    / visibility_summary["Total_Rows"]
    * 100
)

visibility_summary

,Total_Rows,Missing,Mean,Median,Min,Max,Missing_Percent
Weather,,,,,,,
Clear,114802,0,9996.097850,9998.85,1258.5,15000.0,0.000000
Fog,17715,1021,249.683731,250.10,66.4,458.5,5.763477
Rain,43883,2675,2005.025073,2003.40,483.5,3736.2,6.095755
Snow,23003,1383,1000.793363,1000.50,202.0,1776.8,6.012259


In [ ]:
df_clean["Visibility_m"] = (
    df_clean["Visibility_m"]
    .fillna(
        df_clean.groupby("Weather")["Visibility_m"]
        .transform("median")
    )
)

In [ ]:
df_clean["Visibility_m"].isna().sum()

np.int64(0)

In [ ]:
df_clean.groupby("Weather")["Visibility_m"].median()

Weather
Clear    9998.85
Fog       250.10
Rain     2003.40
Snow     1000.50
Name: Visibility_m, dtype: float64

## 9. Impute Missing Driver Experience

`Driver_Experience_yrs` contains missing observations and is logically related to driver age.

The relationship between age and driving experience is examined before imputation. Existing records are also checked to ensure that reported driving experience does not exceed driver age.

An implied driving-start age is temporarily calculated as a diagnostic measure. Missing driver-experience values are then filled using the median experience of drivers with the same age rather than using one global median.

In [ ]:
df_clean[["Driver_Age", "Driver_Experience_yrs"]].corr()

,Driver_Age,Driver_Experience_yrs
Driver_Age,1.000000,0.425591
Driver_Experience_yrs,0.425591,1.000000


In [ ]:
df_clean.loc[
    df_clean["Driver_Experience_yrs"].isna(),
    ["Driver_Age", "Driver_Experience_yrs"]
].head(10)

,Driver_Age,Driver_Experience_yrs
57,49.0,NaN
236,36.0,NaN
257,51.0,NaN
277,55.0,NaN
338,26.0,NaN
352,53.0,NaN
380,28.0,NaN
383,27.0,NaN
476,62.0,NaN
493,50.0,NaN


In [ ]:
df_clean[["Driver_Age", "Driver_Experience_yrs"]].describe()

,Driver_Age,Driver_Experience_yrs
count,199403.000000,195417.000000
mean,38.350737,10.609025
std,13.290118,7.795888
min,16.000000,0.000000
25%,29.000000,4.700000
50%,38.000000,9.400000
75%,47.000000,15.200000
max,85.000000,59.000000


In [ ]:
df_clean[
    df_clean["Driver_Experience_yrs"].notna()
    & (df_clean["Driver_Experience_yrs"] > df_clean["Driver_Age"])
][["Driver_Age", "Driver_Experience_yrs"]]

,Driver_Age,Driver_Experience_yrs


In [ ]:
df_clean["Implied_Driving_Start_Age"] = (
    df_clean["Driver_Age"]
    - df_clean["Driver_Experience_yrs"]
)

In [ ]:
df_clean["Implied_Driving_Start_Age"].describe()

count    195417.000000
mean         27.741425
std          12.218828
min          16.000000
25%          16.000000
50%          24.800000
75%          35.800000
max          85.000000
Name: Implied_Driving_Start_Age, dtype: float64

In [ ]:
df_clean[
    df_clean["Implied_Driving_Start_Age"] < 16
][
    ["Driver_Age", "Driver_Experience_yrs", "Implied_Driving_Start_Age"]
].head(20)

,Driver_Age,Driver_Experience_yrs,Implied_Driving_Start_Age


In [ ]:
experience_by_age = (
    df_clean.groupby("Driver_Age")["Driver_Experience_yrs"]
    .agg(
        Total_Rows="size",
        Missing=lambda x: x.isna().sum(),
        Median_Experience="median"
    )
)

experience_by_age["Missing_Percent"] = (
    experience_by_age["Missing"]
    / experience_by_age["Total_Rows"]
    * 100
)

experience_by_age

,Total_Rows,Missing,Median_Experience,Missing_Percent
Driver_Age,,,,
16.0,12331,223,0.00,1.808450
17.0,1871,43,1.00,2.298236
18.0,2077,42,2.00,2.022147
19.0,2201,44,3.00,1.999091
20.0,2495,44,4.00,1.763527
...,...,...,...,...
81.0,60,2,13.10,3.333333
82.0,32,0,14.25,0.000000
83.0,38,0,11.25,0.000000


In [ ]:
df_clean["Driver_Experience_yrs"] = (
    df_clean["Driver_Experience_yrs"]
    .fillna(
        df_clean.groupby("Driver_Age")["Driver_Experience_yrs"]
        .transform("median")
    )
)

In [ ]:
df_clean["Driver_Experience_yrs"].isna().sum()

np.int64(0)

In [ ]:
df_clean[
    df_clean["Driver_Experience_yrs"] > df_clean["Driver_Age"]
]

,Record_ID,Timestamp,Date,Hour,Month,Day_of_Week,Time_of_Day,Is_Holiday,Area_Type,Road_Type,...,Recorded_Speed_kmh,Baseline_Expected_Response_Time_min,Accident_Occurred,Collision_Type,Vehicles_Involved,Accident_Severity,Dispatch_Priority,Actual_Response_Time_min,Road_Closure_Duration_min,Implied_Driving_Start_Age


In [ ]:
df_clean["Implied_Driving_Start_Age"] = (
    df_clean["Driver_Age"]
    - df_clean["Driver_Experience_yrs"]
)

In [ ]:
df_clean["Implied_Driving_Start_Age"].describe()

count    199403.000000
mean         27.745950
std          12.198011
min          16.000000
25%          16.000000
50%          24.800000
75%          35.800000
max          85.000000
Name: Implied_Driving_Start_Age, dtype: float64

## 10. Convert and Validate Date/Time Fields

`Timestamp` and `Date` are initially stored as strings and are converted to Pandas datetime values.

After conversion, the timestamp is validated against the existing:
- `Date`
- `Hour`
- `Month`
- `Day_of_Week`

This confirms that the related time fields are internally consistent.

In [ ]:
df_clean[
    ["Timestamp", "Date", "Hour", "Month", "Day_of_Week"]
].head(10)

,Timestamp,Date,Hour,Month,Day_of_Week
0,2024-03-14 17:35:06.841829282,2024-03-14,17,3,Thursday
1,2023-05-25 08:10:56.398060108,2023-05-25,8,5,Thursday
2,2023-11-07 08:25:17.721608262,2023-11-07,8,11,Tuesday
3,2024-07-24 09:30:00.947374674,2024-07-24,9,7,Wednesday
4,2024-04-03 09:34:07.316651267,2024-04-03,9,4,Wednesday
5,2022-12-30 01:16:47.065718381,2022-12-30,1,12,Friday
6,2025-04-03 09:56:16.240275423,2025-04-03,9,4,Thursday
7,2022-09-14 09:39:13.770496708,2022-09-14,9,9,Wednesday
8,2022-05-28 07:28:00.317882302,2022-05-28,7,5,Saturday
9,2025-11-08 19:24:33.007032736,2025-11-08,19,11,Saturday


In [ ]:
df_clean["Timestamp"] = pd.to_datetime(
    df_clean["Timestamp"],
    errors="coerce"
)

df_clean["Date"] = pd.to_datetime(
    df_clean["Date"],
    errors="coerce"
)

In [ ]:
df_clean[["Timestamp", "Date"]].dtypes

Timestamp    datetime64[ns]
Date         datetime64[us]
dtype: object

In [ ]:
df_clean[["Timestamp", "Date"]].isna().sum()

Timestamp    0
Date         0
dtype: int64

In [ ]:
date_mismatch = (
    df_clean["Timestamp"].dt.normalize()
    != df_clean["Date"]
).sum()

hour_mismatch = (
    df_clean["Timestamp"].dt.hour
    != df_clean["Hour"]
).sum()

month_mismatch = (
    df_clean["Timestamp"].dt.month
    != df_clean["Month"]
).sum()

day_mismatch = (
    df_clean["Timestamp"].dt.day_name()
    != df_clean["Day_of_Week"]
).sum()

print("Date mismatches:", date_mismatch)
print("Hour mismatches:", hour_mismatch)
print("Month mismatches:", month_mismatch)
print("Day-of-week mismatches:", day_mismatch)

Date mismatches: 0
Hour mismatches: 0
Month mismatches: 0
Day-of-week mismatches: 0


## 11. Feature Engineering

A new `Year` column is derived from the cleaned `Date` field.

This creates an explicit yearly time dimension that can be used by the EDA and dashboard teams for grouping, filtering, and trend analysis.

In [ ]:
df_clean["Year"] = df_clean["Date"].dt.year

df_clean["Year"].value_counts().sort_index()

Year
2022    49603
2023    49660
2024    50312
2025    49828
Name: count, dtype: int64

## 12. Numeric Range and Sanity Checks

Key numeric fields are reviewed for implausible values, unexpected ranges, and possible sentinel values.

Most numeric fields fall within reasonable ranges. However, `Recorded_Speed_kmh` contains values of `999.0 km/h`, which are physically implausible and repeated across multiple records.

These values are investigated separately before correction.

In [ ]:
numeric_columns = [
    "Speed_Limit_kmh",
    "Visibility_m",
    "Congestion_Index",
    "Driver_Age",
    "Driver_Experience_yrs",
    "Vehicle_Age_yrs",
    "Recorded_Speed_kmh",
    "Baseline_Expected_Response_Time_min",
    "Vehicles_Involved",
    "Actual_Response_Time_min",
    "Road_Closure_Duration_min"
]

df_clean[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
Speed_Limit_kmh,199403.0,57.469998,26.352383,30.0,30.0,50.0,80.0,120.0
Visibility_m,199403.0,6333.899740,4553.160233,66.4,1701.5,7759.0,10336.0,15000.0
Congestion_Index,199403.0,55.638528,28.714897,2.2,31.6,53.7,79.1,100.0
Driver_Age,199403.0,38.350737,13.290118,16.0,29.0,38.0,47.0,85.0
Driver_Experience_yrs,199403.0,10.604787,7.732624,0.0,4.7,9.6,15.0,59.0
Vehicle_Age_yrs,199403.0,6.980259,4.769368,0.0,3.4,6.1,9.4,30.0
Recorded_Speed_kmh,199403.0,49.601987,31.937535,0.0,30.4,43.9,62.2,999.0
Baseline_Expected_Response_Time_min,199403.0,12.383709,5.491378,3.0,8.4,11.0,14.3,29.6
Vehicles_Involved,199403.0,0.086443,0.446397,0.0,0.0,0.0,0.0,5.0
Actual_Response_Time_min,199403.0,0.904273,4.663120,0.0,0.0,0.0,0.0,56.6


In [ ]:
df_clean["Recorded_Speed_kmh"].nlargest(20)

69       999.0
2154     999.0
2259     999.0
3600     999.0
9735     999.0
10679    999.0
12574    999.0
12618    999.0
13021    999.0
17695    999.0
18294    999.0
19295    999.0
23900    999.0
25879    999.0
27023    999.0
27280    999.0
27676    999.0
27969    999.0
29473    999.0
33273    999.0
Name: Recorded_Speed_kmh, dtype: float64

In [ ]:
(df_clean["Recorded_Speed_kmh"] > 200).sum()

np.int64(104)

In [ ]:
df_clean.loc[
    df_clean["Recorded_Speed_kmh"] > 200,
    [
        "Record_ID",
        "Speed_Limit_kmh",
        "Recorded_Speed_kmh",
        "Road_Type",
        "Accident_Occurred",
        "Accident_Severity"
    ]
].sort_values(
    "Recorded_Speed_kmh",
    ascending=False
)

,Record_ID,Speed_Limit_kmh,Recorded_Speed_kmh,Road_Type,Accident_Occurred,Accident_Severity
69,REC_0000069,60,999.0,Arterial,0,No Accident
2154,REC_0002154,90,999.0,Interstate,0,No Accident
2259,REC_0002259,30,999.0,Local,0,No Accident
3600,REC_0003600,50,999.0,Collector,0,No Accident
9735,REC_0009735,60,999.0,Collector,0,No Accident
...,...,...,...,...,...,...
186310,REC_0186310,30,999.0,Local,0,No Accident
186715,REC_0186715,30,999.0,Local,0,No Accident
186843,REC_0186843,30,999.0,Local,0,No Accident
193830,REC_0193830,90,999.0,Interstate,0,No Accident


In [ ]:
df_clean.loc[
    df_clean["Recorded_Speed_kmh"] > 200,
    "Recorded_Speed_kmh"
].value_counts()

Recorded_Speed_kmh
999.0    104
Name: count, dtype: int64

In [ ]:
df_clean.loc[
    df_clean["Recorded_Speed_kmh"] > 200,
    "Speed_Limit_kmh"
].value_counts().sort_index()

Speed_Limit_kmh
30     33
40      3
50     21
60     23
80      4
90     10
100     8
120     2
Name: count, dtype: int64

In [ ]:
df_clean.loc[
    df_clean["Recorded_Speed_kmh"] > 200,
    [
        "Recorded_Speed_kmh",
        "Speed_Limit_kmh",
        "Weather",
        "Road_Condition",
        "Accident_Occurred"
    ]
].head(20)

,Recorded_Speed_kmh,Speed_Limit_kmh,Weather,Road_Condition,Accident_Occurred
69,999.0,60,Clear,Dry,0
2154,999.0,90,Fog,Dry,0
2259,999.0,30,Clear,Dry,0
3600,999.0,50,Clear,Dry,0
9735,999.0,60,Snow,Snowy,0
10679,999.0,40,Clear,Dry,0
12574,999.0,50,Rain,Wet,1
12618,999.0,50,Fog,Dry,0
13021,999.0,30,Clear,Dry,0
17695,999.0,30,Clear,Dry,0


In [ ]:
df_clean.loc[
    df_clean["Recorded_Speed_kmh"] == 999.0,
    "Recorded_Speed_kmh"
] = pd.NA

In [ ]:
speed_by_limit = (
    df_clean.groupby("Speed_Limit_kmh")["Recorded_Speed_kmh"]
    .agg(
        Total_Rows="size",
        Missing=lambda x: x.isna().sum(),
        Mean="mean",
        Median="median",
        Min="min",
        Max="max"
    )
)

speed_by_limit

,Total_Rows,Missing,Mean,Median,Min,Max
Speed_Limit_kmh,,,,,,
30,57780,33,27.608542,28.0,0.0,47.437908
40,12197,3,33.831736,34.1,0.0,59.481929
50,43897,21,42.243287,42.5,0.0,76.296749
60,31774,23,53.172336,53.8,0.0,86.900000
80,15818,4,74.009910,74.8,0.0,112.100000
90,12186,10,66.801533,65.7,0.0,124.800000
100,15759,8,77.848953,76.7,0.0,137.700000
120,9992,2,102.935529,104.0,0.0,160.736211


In [ ]:
df_clean["Recorded_Speed_kmh"] = (
    df_clean["Recorded_Speed_kmh"]
    .fillna(
        df_clean.groupby("Speed_Limit_kmh")["Recorded_Speed_kmh"]
        .transform("median")
    )
)

In [ ]:
df_clean["Recorded_Speed_kmh"].isna().sum()

np.int64(0)

In [ ]:
(df_clean["Recorded_Speed_kmh"] == 999.0).sum()

np.int64(0)

In [ ]:
(df_clean["Recorded_Speed_kmh"] == 0).sum()

np.int64(122)

In [ ]:
df_clean.loc[
    df_clean["Recorded_Speed_kmh"] == 0,
    [
        "Record_ID",
        "Speed_Limit_kmh",
        "Congestion_Index",
        "Time_of_Day",
        "Road_Condition",
        "Accident_Occurred"
    ]
].head(20)

,Record_ID,Speed_Limit_kmh,Congestion_Index,Time_of_Day,Road_Condition,Accident_Occurred
3720,REC_0003720,60,29.4,Morning Peak,Wet,0
4619,REC_0004619,30,61.9,Evening Peak,Snowy,0
8738,REC_0008738,40,68.0,Evening Peak,Snowy,0
9431,REC_0009431,90,86.3,Midday,Dry,0
13842,REC_0013842,30,49.0,Morning Peak,Icy,0
15163,REC_0015163,30,41.5,Midday,Wet,0
15487,REC_0015487,50,98.7,Morning Peak,Wet,0
17197,REC_0017197,60,79.5,Evening Peak,Wet,0
17925,REC_0017925,50,39.4,Midday,Wet,0
18637,REC_0018637,120,75.6,Morning Peak,Dry,0


## 13. Validate Accident and Emergency-Response Fields

Accident-related numeric and categorical fields are checked for logical consistency.

For non-accident records, the dataset should contain:
- zero vehicles involved
- zero actual response time
- zero road-closure duration
- `No Accident` collision classification
- `Not Applicable` dispatch priority

Accident records should contain valid vehicle counts, accident severity, response times, and closure durations.

These checks confirm that the related fields remain internally consistent after cleaning.

In [ ]:
df_clean.groupby("Accident_Occurred").agg(
    Rows=("Accident_Occurred", "size"),
    Avg_Vehicles=("Vehicles_Involved", "mean"),
    Min_Vehicles=("Vehicles_Involved", "min"),
    Max_Vehicles=("Vehicles_Involved", "max"),
    Avg_Response_Time=("Actual_Response_Time_min", "mean"),
    Min_Response_Time=("Actual_Response_Time_min", "min"),
    Max_Response_Time=("Actual_Response_Time_min", "max"),
    Avg_Road_Closure=("Road_Closure_Duration_min", "mean"),
    Min_Road_Closure=("Road_Closure_Duration_min", "min"),
    Max_Road_Closure=("Road_Closure_Duration_min", "max")
)

,Rows,Avg_Vehicles,Min_Vehicles,Max_Vehicles,Avg_Response_Time,Min_Response_Time,Max_Response_Time,Avg_Road_Closure,Min_Road_Closure,Max_Road_Closure
Accident_Occurred,,,,,,,,,,
0,191190,0.000000,0,0,0.000000,0.0,0.0,0.000000,0.0,0.0
1,8213,2.098746,1,5,21.954791,4.8,56.6,186.057044,55.6,1772.5


In [ ]:
accident_consistency = pd.DataFrame({
    "No_Accident_with_Collision_Type": [
        (
            (df_clean["Accident_Occurred"] == 0)
            & (df_clean["Collision_Type"] != "No Accident")
        ).sum()
    ],
    "No_Accident_with_Dispatch_Priority": [
        (
            (df_clean["Accident_Occurred"] == 0)
            & (df_clean["Dispatch_Priority"] != "Not Applicable")
        ).sum()
    ],
    "Accident_with_No_Accident_Severity": [
        (
            (df_clean["Accident_Occurred"] == 1)
            & (df_clean["Accident_Severity"] == "No Accident")
        ).sum()
    ],
    "Accident_with_Zero_Vehicles": [
        (
            (df_clean["Accident_Occurred"] == 1)
            & (df_clean["Vehicles_Involved"] == 0)
        ).sum()
    ]
})

accident_consistency

,No_Accident_with_Collision_Type,No_Accident_with_Dispatch_Priority,Accident_with_No_Accident_Severity,Accident_with_Zero_Vehicles
0,0,0,0,0


## 14. Remove Temporary Diagnostic Fields

`Implied_Driving_Start_Age` was created only to validate the relationship between driver age and driving experience.

Because it is not required for the final analysis and was not part of the original dataset, the temporary diagnostic column is removed before export.

In [ ]:
df_clean = df_clean.drop(columns=["Implied_Driving_Start_Age"])

## 15. Final Data Quality Validation

A final validation is performed to confirm that the cleaned dataset contains:

- no missing values
- no duplicate records
- unique `Record_ID` values
- valid data types
- the expected number of cleaned rows and columns

At this stage, the dataset should be ready for exploratory data analysis, visualization, and dashboard development.

In [ ]:
print("Shape:", df_clean.shape)
print("Missing values:", df_clean.isna().sum().sum())
print("Duplicate rows:", df_clean.duplicated().sum())
print("Unique Record_IDs:", df_clean["Record_ID"].nunique())
print("Total rows:", len(df_clean))

Shape: (199403, 30)
Missing values: 0
Duplicate rows: 0
Unique Record_IDs: 199403
Total rows: 199403


In [ ]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 199403 entries, 0 to 199402
Data columns (total 30 columns):
 #   Column                               Non-Null Count   Dtype         
---  ------                               --------------   -----         
 0   Record_ID                            199403 non-null  str           
 1   Timestamp                            199403 non-null  datetime64[ns]
 2   Date                                 199403 non-null  datetime64[us]
 3   Hour                                 199403 non-null  int64         
 4   Month                                199403 non-null  int64         
 5   Day_of_Week                          199403 non-null  str           
 6   Time_of_Day                          199403 non-null  str           
 7   Is_Holiday                           199403 non-null  int64         
 8   Area_Type                            199403 non-null  str           
 9   Road_Type                            199403 non-null  str           
 10  Interse

## 16. Export and Verify the Cleaned Dataset

The cleaned DataFrame is exported as:

`synthetic_traffic_benchmark_200k_cleaned.csv`

The exported file is then reloaded with the date columns parsed correctly and checked again for:

- dataset shape
- missing values
- duplicate rows
- unique record identifiers
- datetime types

This final verification confirms that the exported file preserves the cleaned dataset correctly for handoff to the rest of the project team.

In [ ]:
df_clean.to_csv(
    "synthetic_traffic_benchmark_200k_cleaned.csv",
    index=False
)

In [ ]:
df_check = pd.read_csv(
    "synthetic_traffic_benchmark_200k_cleaned.csv",
    parse_dates=["Timestamp", "Date"]
)

print("Shape:", df_check.shape)
print("Missing values:", df_check.isna().sum().sum())
print("Duplicate rows:", df_check.duplicated().sum())
print("Unique Record_IDs:", df_check["Record_ID"].nunique())
print(df_check[["Timestamp", "Date"]].dtypes)

Shape: (199403, 30)
Missing values: 0
Duplicate rows: 0
Unique Record_IDs: 199403
Timestamp    datetime64[ns]
Date         datetime64[us]
dtype: object


## Data Wrangling Complete

The original dataset contained 200,000 rows and 29 columns.

After cleaning:
- 597 exact duplicate records were removed
- inconsistent weather categories were standardized
- structural missing values were converted into explicit categories
- missing visibility values were imputed using weather-specific medians
- missing driver-experience values were imputed using age-specific medians
- invalid `999.0 km/h` speed values were corrected using speed-limit-specific medians
- date and time fields were converted and validated
- a `Year` feature was added
- accident-related fields were validated for logical consistency

The final cleaned dataset is ready for handoff to the EDA, visualization, and dashboard portions of the project.